# Time-Series Trend & Rolling Metrics Analysis

This notebook demonstrates temporal analysis techniques using Pandas:
1. **Resampling Data** into weekly and monthly frequency aggregates.
2. **Computing Rolling Moving Averages** (7-day and 30-day) to smooth daily volatility.
3. **Calculating Month-over-Month Percentage Change** (`.pct_change()`).
4. **Tracking Cumulative Sum** (`.cumsum()`).
5. **Identifying Business Trends and Actions**.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load synthetic 1-year daily revenue time-series data
filepath = '../data/raw/daily_revenue_timeseries.csv'
if not os.path.exists(filepath):
    filepath = 'data/raw/daily_revenue_timeseries.csv'

df = pd.read_csv(filepath)
df['date'] = pd.to_datetime(df['date'])
print(f"Loaded {len(df)} daily records from {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}.")

## Task 1: Resample Data by Time Period (Weekly & Monthly)

In [2]:
df_ts = df.set_index('date')

# Weekly Aggregations
weekly_revenue = df_ts['revenue'].resample('W').sum()
weekly_count = df_ts['orders'].resample('W').count()
weekly_avg = df_ts['revenue'].resample('W').mean()

# Monthly Aggregations
try:
    monthly_revenue = df_ts['revenue'].resample('ME').sum()
except ValueError:
    monthly_revenue = df_ts['revenue'].resample('M').sum()

print('Weekly Revenue Summary (First 5 weeks):')
print(weekly_revenue.head())
print('\nMonthly Revenue Summary:')
print(monthly_revenue)

## Task 2: Compute Rolling Window Averages (7-day vs. 30-day)

In [3]:
df['revenue_ma7'] = df['revenue'].rolling(window=7).mean()
df['revenue_ma30'] = df['revenue'].rolling(window=30).mean()

plt.figure(figsize=(12, 6))
plt.plot(df['date'], df['revenue'], label='Raw Daily Revenue', alpha=0.3, color='gray')
plt.plot(df['date'], df['revenue_ma7'], label='7-Day Moving Average', color='blue', linewidth=1.5)
plt.plot(df['date'], df['revenue_ma30'], label='30-Day Moving Average', color='red', linewidth=2.0)
plt.title('Daily Revenue vs. 7-Day & 30-Day Moving Averages')
plt.xlabel('Date')
plt.ylabel('Revenue ($)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## Task 3: Calculate Month-over-Month Percentage Change (`.pct_change()`)

In [4]:
mom_change = monthly_revenue.pct_change() * 100
print('Month-over-Month Growth Rates (%):')
for idx, val in mom_change.items():
    if pd.isna(val):
        print(f"{idx.strftime('%Y-%m')}: Baseline Month")
    else:
        print(f"{idx.strftime('%Y-%m')}: {val:+.2f}%")

## Task 4: Compute Cumulative Accumulated Revenue (`.cumsum()`)

In [5]:
df['cumulative_revenue'] = df['revenue'].cumsum()
total_accumulated = df['cumulative_revenue'].iloc[-1]
print(f"Total Accumulated Revenue: ${total_accumulated:,.2f}")

plt.figure(figsize=(10, 5))
plt.plot(df['date'], df['cumulative_revenue'], color='green', linewidth=2.0)
plt.title('Cumulative Revenue Over Time')
plt.xlabel('Date')
plt.ylabel('Cumulative Revenue ($)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## Task 5: Trend Analysis & Strategic Business Implications

In [6]:
recent_ma30 = df['revenue_ma30'].dropna().iloc[-30:]
first_ma30 = recent_ma30.iloc[0]
last_ma30 = recent_ma30.iloc[-1]
trend_direction = 'UPTREND' if last_ma30 > first_ma30 else 'DOWNTREND'
trend_magnitude = ((last_ma30 - first_ma30) / first_ma30) * 100

print(f"Trend Direction: {trend_direction}")
print(f"30-Day Growth Magnitude: {trend_magnitude:+.2f}%")
print(f"Daily Volatility: ${df['revenue'].std():,.2f}")